# MIT Paired 10-Seed Comparison: CQRF+CBR vs QRR+CBR

This notebook performs a **strictly paired** comparison on the MIT battery dataset.

## Controlled settings

- Seeds: `0–9`
- Split: identical 60/20/20 cell-disjoint split for CQRF and QRR within each seed
- Reservoir: same 10-qubit topology, frozen parameters, input projection, coupling strength, and 57-observable order
- Readout for both models: **CBR = 57 → 2 → 2 → 1**, **Tanh**, no dropout, **125 trainable parameters**
- QRR memory is reset at every cell boundary
- The first five cycles are processed as QRR warmup and then discarded **for every cell**
- CQRF is evaluated on exactly the same post-warmup cycles
- Raw-feature normalization is fitted only on post-warmup training-cell cycles
- CBR feature normalization is fitted only on the training reservoir features
- External cell-disjoint validation cells are used for early stopping
- Seed-wise splits, predictions, checkpoints, histories, runtimes, feature caches, and paired statistics are saved

## Important

This corrected protocol is intentionally stricter than the older MIT notebooks. Results may differ from the submitted manuscript because the earlier QRR notebook used ReLU and allowed sequence concatenation across cells.

In [1]:
# ================================================================
# 1. Environment and paths
# ================================================================
import os, sys, subprocess, time, json, warnings, pickle, datetime, copy, glob, random, gc
from pathlib import Path

def _ensure_packages():
    required = {
        "pennylane": "pennylane",
        "scipy": "scipy",
        "psutil": "psutil",
        "sklearn": "scikit-learn",
    }
    missing = []
    for module_name, package_name in required.items():
        try:
            __import__(module_name)
        except Exception:
            missing.append(package_name)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

_ensure_packages()

import numpy as np
import pandas as pd
import pennylane as qml
import psutil

from scipy.stats import wilcoxon
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# Mount Google Drive only when running in Colab.
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/HB/QRC_Battery_SOH"
MIT_PATH = os.path.join(PROJECT_ROOT, "data/processed/MIT")

EXP_ID = "789_EXP-MIT-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-v1"
PROTOCOL_TAG = "paired_v1_per_cell_warmup_external_val_tanh"

CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints", EXP_ID)
RES_DIR = os.path.join(PROJECT_ROOT, "results", EXP_ID)
LOG_DIR = os.path.join(PROJECT_ROOT, "logs", EXP_ID)
FEAT_DIR = os.path.join(PROJECT_ROOT, "features", EXP_ID)
SPLIT_DIR = os.path.join(RES_DIR, "splits")
PRED_DIR = os.path.join(RES_DIR, "predictions")
HIST_DIR = os.path.join(RES_DIR, "histories")
RUNTIME_DIR = os.path.join(RES_DIR, "runtimes")

for directory in [CKPT_DIR, RES_DIR, LOG_DIR, FEAT_DIR, SPLIT_DIR, PRED_DIR, HIST_DIR, RUNTIME_DIR]:
    os.makedirs(directory, exist_ok=True)

LOG_PATH = os.path.join(LOG_DIR, f"{EXP_ID}.log")

def log(message):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MIT_PATH:", MIT_PATH, "exists=", os.path.exists(MIT_PATH))
print("EXP_ID:", EXP_ID)
print("Results:", RES_DIR)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/HB/QRC_Battery_SOH
MIT_PATH: /content/drive/MyDrive/HB/QRC_Battery_SOH/data/processed/MIT exists= True
EXP_ID: 789_EXP-MIT-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-v1
Results: /content/drive/MyDrive/HB/QRC_Battery_SOH/results/789_EXP-MIT-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-v1


In [2]:
# ================================================================
# 2. Frozen protocol configuration
# ================================================================
RUN_MODE = "FULL"      # "SMOKE" runs seed 0 only. Change to "FULL" after validation.
SEEDS = [7, 8, 9]
#SMOKE_SEEDS = [4]

WARMUP = 5
RESERVOIR_SEED = 42
COUPLING_STRENGTH = 0.3
N_MEMORY = 4
N_PROCESSOR = 6
N_RESERVOIR_LAYERS = 2

USE_FEATURE_CACHE = True
REGENERATE_FEATURES = False
SKIP_COMPLETED = True

# Shared CBR settings for both CQRF and QRR.
CBR_HIDDEN_DIMS = (2, 2)
CBR_DROPOUT = 0.0
CBR_EPOCHS = 300
CBR_LR = 1e-3
CBR_BATCH_SIZE = 64
CBR_PATIENCE = 40
CBR_WEIGHT_DECAY = 1e-5
CBR_GRAD_CLIP = 1.0

# Bootstrap configuration for paired seed statistics.
BOOTSTRAP_RESAMPLES = 10000
BOOTSTRAP_SEED = 2026

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def count_dense_params(input_dim, hidden_dims):
    dims = [input_dim, *hidden_dims, 1]
    return sum(dims[i] * dims[i + 1] + dims[i + 1] for i in range(len(dims) - 1))

EXPECTED_CBR_PARAMS = count_dense_params(57, CBR_HIDDEN_DIMS)
assert EXPECTED_CBR_PARAMS == 125

ACTIVE_SEEDS = SMOKE_SEEDS if RUN_MODE.upper() == "SMOKE" else SEEDS

print("RUN_MODE:", RUN_MODE)
print("ACTIVE_SEEDS:", ACTIVE_SEEDS)
print("Device:", device)
print("Shared CBR: 57 -> 2 -> 2 -> 1, Tanh, dropout=0.0, params=", EXPECTED_CBR_PARAMS)

RUN_MODE: FULL
ACTIVE_SEEDS: [7, 8, 9]
Device: cpu
Shared CBR: 57 -> 2 -> 2 -> 1, Tanh, dropout=0.0, params= 125


In [3]:
# ================================================================
# 3. Shared reservoir implementation
#    The only CQRF/QRR difference is the recurrent memory update.
# ================================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


class CoupledQuantumReservoir10q:
    """
    recurrent=False -> CQRF: fixed zero auxiliary RY angles for every cycle.
    recurrent=True  -> QRR: memory angles updated from the previous cycle's Z expectations.

    Both variants share exactly the same:
      - frozen reservoir parameters
      - frozen input projection
      - fixed auxiliary RZ rotations
      - topology and gate order
      - 57-observable order
    """
    def __init__(
        self,
        n_inputs,
        recurrent,
        n_memory=N_MEMORY,
        n_processor=N_PROCESSOR,
        n_reservoir_layers=N_RESERVOIR_LAYERS,
        seed=RESERVOIR_SEED,
        coupling_strength=COUPLING_STRENGTH,
    ):
        self.n_memory = int(n_memory)
        self.n_processor = int(n_processor)
        self.n_qubits = self.n_memory + self.n_processor
        self.n_reservoir_layers = int(n_reservoir_layers)
        self.n_inputs = int(n_inputs)
        self.seed = int(seed)
        self.coupling_strength = float(coupling_strength)
        self.recurrent = bool(recurrent)

        self.memory_wires = list(range(self.n_memory))
        self.proc_wires = list(range(self.n_memory, self.n_qubits))
        self.proc_pairs = [(i, j) for i in self.proc_wires for j in self.proc_wires if i < j]
        self.cross_pairs = [(i, j) for i in self.memory_wires for j in self.proc_wires]

        self.n_single = self.n_processor * 3
        self.n_proc_pairs = len(self.proc_pairs)
        self.n_cross_pairs = len(self.cross_pairs)
        self.n_outputs = self.n_single + self.n_proc_pairs + self.n_cross_pairs
        assert self.n_outputs == 57

        rng = np.random.RandomState(self.seed)
        self.reservoir_params = rng.uniform(
            0, 2 * np.pi, size=(self.n_reservoir_layers, self.n_qubits, 2)
        )
        self.input_projection = rng.randn(self.n_inputs, self.n_processor) * 0.5
        self.memory_rotation = rng.uniform(0, 2 * np.pi, size=(self.n_memory, 2))

        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.memory_angles = np.zeros(self.n_memory, dtype=np.float64)
        self._build_circuit()

    @property
    def model_name(self):
        return "QRR+CBR" if self.recurrent else "CQRF+CBR"

    def _build_circuit(self):
        n_q = self.n_qubits
        n_m = self.n_memory
        n_p = self.n_processor
        n_l = self.n_reservoir_layers
        rp = self.reservoir_params
        mw = self.memory_wires
        pw = self.proc_wires
        pp = self.proc_pairs
        cp = self.cross_pairs
        coupling = self.coupling_strength
        memory_rotation = self.memory_rotation

        @qml.qnode(self.dev)
        def circuit(projected_input, memory_angles):
            # Auxiliary / memory register.
            for i, wire in enumerate(mw):
                qml.RY(memory_angles[i], wires=wire)
                qml.RZ(memory_rotation[i, 0], wires=wire)

            # Processor input encoding.
            for i, wire in enumerate(pw):
                qml.RY(projected_input[i], wires=wire)
                qml.RZ(projected_input[i] * 0.5, wires=wire)

            # Within-register entanglement.
            for i in range(n_m - 1):
                qml.CNOT(wires=[mw[i], mw[i + 1]])
            for i in range(n_p - 1):
                qml.CNOT(wires=[pw[i], pw[i + 1]])
            qml.CNOT(wires=[pw[-1], pw[0]])

            # Direct, reverse, and skip cross-register coupling.
            for i in range(n_m):
                qml.CRY(coupling * np.pi, wires=[mw[i], pw[i]])
                qml.CRY(coupling * 0.5 * np.pi, wires=[pw[i], mw[i]])
            for i in range(n_m):
                qml.CRY(coupling * 0.3 * np.pi, wires=[mw[i], pw[(i + 2) % n_p]])

            # Frozen reservoir dynamics.
            for layer in range(n_l):
                for i in range(n_q):
                    qml.RY(rp[layer, i, 0], wires=i)
                    qml.RZ(rp[layer, i, 1], wires=i)
                for i in range(n_q):
                    qml.CNOT(wires=[i, (i + 1) % n_q])
                for i in range(0, n_q - 2, 2):
                    qml.CNOT(wires=[i, i + 2])

            observables = []
            for wire in pw:
                observables.extend(
                    [
                        qml.expval(qml.PauliX(wire)),
                        qml.expval(qml.PauliY(wire)),
                        qml.expval(qml.PauliZ(wire)),
                    ]
                )
            for i, j in pp:
                observables.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)))
            for i, j in cp:
                observables.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)))
            for wire in mw:
                observables.append(qml.expval(qml.PauliZ(wire)))
            return observables

        self._circuit = circuit

    def reset(self):
        self.memory_angles = np.zeros(self.n_memory, dtype=np.float64)

    def process_single(self, x):
        projected_input = np.clip(x @ self.input_projection, -np.pi, np.pi)
        outputs = np.asarray(self._circuit(projected_input, self.memory_angles), dtype=np.float64)
        features = outputs[: self.n_outputs].astype(np.float32)

        if self.recurrent:
            memory_readout = outputs[self.n_outputs :]
            self.memory_angles = (memory_readout + 1.0) * np.pi
        else:
            # CQRF remains fixed for every cycle.
            self.memory_angles.fill(0.0)

        return features

    def process_cell(self, x_cell):
        # Critical fairness rule: memory never crosses a cell boundary.
        self.reset()
        states = np.zeros((len(x_cell), self.n_outputs), dtype=np.float32)
        for index, x in enumerate(x_cell):
            states[index] = self.process_single(x)
        return states


def normalize_raw_features(features, stats):
    normalized = (features - stats["mean"]) / stats["std"]
    return np.clip(normalized, -3, 3) * (np.pi / 3)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None)))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "MAPE": float(mape),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "R2": float(r2),
        "DerivedScore": float((1.0 - mape) * 100.0),
    }


def current_rss_mb():
    return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))

In [4]:
# ═══════════════════════════════════════════════════════════════════
# MIT dataset loader — robust one-NPZ-per-cell loader.
#
# Fix for MIT files like:
#   2017-05-12_cell000.npz
# where X has shape [cycles, features] but cell_id is scalar / length-1.
# In that case, the file itself is treated as ONE cell.
#
# Expected final structure:
#   mit_dataset[cell_id] = {"features": np.ndarray [cycles, n_inputs], "soh": np.ndarray [cycles]}
# ═══════════════════════════════════════════════════════════════════

TARGET_KEYS = ["soh", "SOH", "y", "target", "targets", "label", "labels", "capacity", "cap"]
FEATURE_KEYS = ["features", "feature", "X", "x", "inputs", "data"]
CELL_ID_KEYS = ["cell_id", "cell", "battery_id", "battery", "id"]


def _unwrap_np_object(x):
    """Unwrap scalar object arrays commonly stored in npz/pkl files."""
    if isinstance(x, np.ndarray) and x.dtype == object and x.shape == ():
        try:
            return x.item()
        except Exception:
            return x
    return x


def _as_array(x):
    x = _unwrap_np_object(x)
    arr = np.asarray(x)
    if arr.dtype == object:
        try:
            arr = np.array(arr.tolist())
        except Exception:
            pass
    return arr


def _clean_features_soh(features, soh, min_cycles=5):
    features = _as_array(features)
    soh = _as_array(soh).reshape(-1)

    # Convert features safely to 2-D numeric array.
    features = np.asarray(features)
    if features.ndim == 1:
        features = features.reshape(-1, 1)
    elif features.ndim > 2:
        features = features.reshape(features.shape[0], -1)

    # If feature orientation is [features, cycles], transpose.
    if features.shape[0] != len(soh) and features.shape[1] == len(soh):
        features = features.T

    try:
        features = features.astype(float)
        soh = soh.astype(float)
    except Exception:
        return None, None

    n = min(features.shape[0], len(soh))
    features, soh = features[:n], soh[:n]

    mask = np.isfinite(soh)
    mask &= np.all(np.isfinite(features), axis=1)
    features, soh = features[mask], soh[mask]
    if len(soh) < min_cycles:
        return None, None

    # If target is capacity-like rather than normalized SOH, normalize by first valid value.
    if np.nanmedian(soh) > 1.5:
        soh = soh / max(float(soh[0]), 1e-8)

    return features, soh


def _scalar_cell_id(x, fallback):
    try:
        arr = np.asarray(_unwrap_np_object(x)).reshape(-1)
        if len(arr) >= 1:
            val = arr[0]
            if isinstance(val, bytes):
                val = val.decode("utf-8", errors="ignore")
            return str(val)
    except Exception:
        pass
    return str(fallback)


def _record_to_cell(cell_id, rec):
    rec = _unwrap_np_object(rec)
    if not isinstance(rec, dict):
        return None
    fkey = next((k for k in FEATURE_KEYS if k in rec), None)
    ykey = next((k for k in TARGET_KEYS if k in rec), None)
    if fkey is None or ykey is None:
        return None
    features, soh = _clean_features_soh(rec[fkey], rec[ykey])
    if features is None:
        return None
    return str(cell_id), {"features": features, "soh": soh, "n_cycles": len(soh)}


def _load_pickle_or_npz_file(path):
    cells = {}
    path = str(path)
    stem = Path(path).stem
    try:
        if path.endswith((".pkl", ".pickle")):
            with open(path, "rb") as f:
                obj = pickle.load(f)
        elif path.endswith(".npz"):
            npz = np.load(path, allow_pickle=True)
            obj = {k: _unwrap_np_object(npz[k]) for k in npz.files}
        else:
            return cells
    except Exception as e:
        print(f"Could not load {path}: {e}")
        return cells

    obj = _unwrap_np_object(obj)

    if isinstance(obj, dict):
        fkey = next((k for k in FEATURE_KEYS if k in obj), None)
        ykey = next((k for k in TARGET_KEYS if k in obj), None)
        ckey = next((k for k in CELL_ID_KEYS if k in obj), None)

        # Direct arrays in one file: one cell OR many cells.
        if fkey and ykey:
            X = _as_array(obj[fkey])
            y = _as_array(obj[ykey]).reshape(-1)

            if ckey is not None:
                cids_raw = _as_array(obj[ckey]).reshape(-1)
                # Split by cell id only when there is one id per cycle.
                if len(cids_raw) == len(y) and (X.ndim == 1 or X.shape[0] == len(y)):
                    for cid in np.unique(cids_raw):
                        m = cids_raw == cid
                        ft, so = _clean_features_soh(X[m], y[m])
                        if ft is not None:
                            cells[str(cid)] = {"features": ft, "soh": so, "n_cycles": len(so)}
                    if cells:
                        return cells
                else:
                    # Important MIT case: one npz per cell, scalar/length-1 cell_id.
                    cid = _scalar_cell_id(obj[ckey], stem)
                    ft, so = _clean_features_soh(X, y)
                    if ft is not None:
                        cells[cid] = {"features": ft, "soh": so, "n_cycles": len(so)}
                    return cells

            # No cell id: treat file as one cell.
            ft, so = _clean_features_soh(X, y)
            if ft is not None:
                cells[stem] = {"features": ft, "soh": so, "n_cycles": len(so)}
            return cells

        # Dict of cell_id -> record.
        for cid, rec in obj.items():
            rec = _unwrap_np_object(rec)
            item = _record_to_cell(cid, rec)
            if item:
                k, v = item
                cells[k] = v
        if cells:
            return cells

    if isinstance(obj, (list, tuple)):
        for i, rec in enumerate(obj):
            rec = _unwrap_np_object(rec)
            if not isinstance(rec, dict):
                continue
            cid = rec.get("cell_id", rec.get("cell", rec.get("battery_id", f"cell_{i:03d}")))
            item = _record_to_cell(cid, rec)
            if item:
                k, v = item
                cells[k] = v
    return cells


def _load_csv_file(path):
    cells = {}
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"Could not read CSV {path}: {e}")
        return cells
    if len(df) < 5:
        return cells

    ckey = next((k for k in CELL_ID_KEYS if k in df.columns), None)
    groups = [(Path(path).stem, df)] if ckey is None else list(df.groupby(ckey))

    for cid, g in groups:
        g = g.copy()
        cyc_cols = [c for c in g.columns if "cycle" in c.lower()]
        if cyc_cols and g[cyc_cols[0]].nunique() < len(g):
            num_cols = g.select_dtypes(include=[np.number]).columns.tolist()
            g = g.groupby(cyc_cols[0])[num_cols].mean().reset_index()

        ycol = next((k for k in TARGET_KEYS if k in g.columns), None)
        if ycol is None:
            cap_candidates = [c for c in g.columns if any(s in c.lower() for s in ["capacity", "cap", "qdis", "q_dis", "discharge"])]
            ycol = cap_candidates[0] if cap_candidates else None
        if ycol is None:
            continue

        numeric_cols = g.select_dtypes(include=[np.number]).columns.tolist()
        exclude = set([ycol]) | set([c for c in numeric_cols if "cycle" in c.lower()])
        feature_cols = [c for c in numeric_cols if c not in exclude]
        if len(feature_cols) == 0:
            continue
        ft, so = _clean_features_soh(g[feature_cols].values, g[ycol].values)
        if ft is not None:
            cells[str(cid)] = {"features": ft, "soh": so, "n_cycles": len(so), "feature_cols": feature_cols}
    return cells


def _inspect_npz_structure(path, max_keys=20):
    try:
        npz = np.load(path, allow_pickle=True)
        print(f"Inspecting {Path(path).name}: keys={npz.files[:max_keys]}")
        for k in npz.files[:max_keys]:
            arr = npz[k]
            print(f"  {k}: shape={arr.shape}, dtype={arr.dtype}")
    except Exception as e:
        print("Could not inspect npz:", e)


def load_mit_dataset(mit_path=MIT_PATH):
    if not os.path.exists(mit_path):
        raise FileNotFoundError(f"MIT_PATH does not exist: {mit_path}")

    all_files = []
    for ext in ["*.pkl", "*.pickle", "*.npz", "*.csv"]:
        all_files.extend(glob.glob(os.path.join(mit_path, "**", ext), recursive=True))
    all_files = sorted(all_files)
    print(f"Found {len(all_files)} candidate MIT files")
    for f in all_files[:20]:
        print("  ", os.path.relpath(f, mit_path))

    dataset = {}
    yielded = []
    for f in all_files:
        if f.endswith((".pkl", ".pickle", ".npz")):
            cells = _load_pickle_or_npz_file(f)
        elif f.endswith(".csv"):
            cells = _load_csv_file(f)
        else:
            cells = {}
        if cells:
            yielded.append((os.path.relpath(f, mit_path), len(cells)))
        for cid, data in cells.items():
            key = cid if cid not in dataset else f"{Path(f).stem}_{cid}"
            dataset[key] = data

    if yielded:
        print("Files that yielded cells (first 20):")
        for name, n in yielded[:20]:
            print(f"  {name}: {n}")

    if not dataset:
        print("No cells extracted. Inspecting first NPZ file for debugging:")
        npz_files = [f for f in all_files if f.endswith('.npz')]
        if npz_files:
            _inspect_npz_structure(npz_files[0])
        raise RuntimeError(
            "Could not auto-load MIT dataset. The loader expected one npz per cell or a processed multi-cell file."
        )

    dims = pd.Series([v["features"].shape[1] for v in dataset.values()])
    common_dim = int(dims.value_counts().idxmax())
    filtered = {k:v for k,v in dataset.items() if v["features"].shape[1] == common_dim and len(v["soh"]) >= 10}
    print(f"Loaded MIT cells before dim filter: {len(dataset)}")
    print(f"Most common feature dimension: {common_dim}; cells kept: {len(filtered)}")

    sample_key = sorted(filtered.keys())[0]
    print("Sample cell:", sample_key, "features", filtered[sample_key]["features"].shape, "soh", filtered[sample_key]["soh"].shape)

    if len(filtered) < 5:
        raise RuntimeError("Too few MIT cells after loading/filtering.")
    return filtered, common_dim


mit_dataset, N_INPUTS = load_mit_dataset(MIT_PATH)
mit_cells = sorted(mit_dataset.keys())
print(f"✅ MIT loaded: {len(mit_cells)} cells, N_INPUTS={N_INPUTS}")
lengths = [len(mit_dataset[c]["soh"]) for c in mit_cells]
print("Cycle length stats: min/median/max =", int(np.min(lengths)), int(np.median(lengths)), int(np.max(lengths)))


Found 140 candidate MIT files
   2017-05-12_cell000.npz
   2017-05-12_cell001.npz
   2017-05-12_cell002.npz
   2017-05-12_cell003.npz
   2017-05-12_cell004.npz
   2017-05-12_cell005.npz
   2017-05-12_cell006.npz
   2017-05-12_cell007.npz
   2017-05-12_cell008.npz
   2017-05-12_cell009.npz
   2017-05-12_cell010.npz
   2017-05-12_cell011.npz
   2017-05-12_cell012.npz
   2017-05-12_cell013.npz
   2017-05-12_cell014.npz
   2017-05-12_cell015.npz
   2017-05-12_cell016.npz
   2017-05-12_cell017.npz
   2017-05-12_cell018.npz
   2017-05-12_cell019.npz
Files that yielded cells (first 20):
  2017-05-12_cell000.npz: 1
  2017-05-12_cell001.npz: 1
  2017-05-12_cell002.npz: 1
  2017-05-12_cell003.npz: 1
  2017-05-12_cell004.npz: 1
  2017-05-12_cell005.npz: 1
  2017-05-12_cell006.npz: 1
  2017-05-12_cell007.npz: 1
  2017-05-12_cell008.npz: 1
  2017-05-12_cell009.npz: 1
  2017-05-12_cell010.npz: 1
  2017-05-12_cell011.npz: 1
  2017-05-12_cell012.npz: 1
  2017-05-12_cell013.npz: 1
  2017-05-12_cell014.

In [5]:
# ================================================================
# 5. Paired cell splits, train-only normalization, per-cell warmup
# ================================================================
def get_mit_split(cells, seed):
    cells = list(cells)
    n_cells = len(cells)
    n_train = int(n_cells * 0.60)
    n_valid = int(n_cells * 0.20)

    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(cells).tolist()

    train_ids = shuffled[:n_train]
    valid_ids = shuffled[n_train : n_train + n_valid]
    test_ids = shuffled[n_train + n_valid :]

    assert not (set(train_ids) & set(valid_ids))
    assert not (set(train_ids) & set(test_ids))
    assert not (set(valid_ids) & set(test_ids))
    assert len(train_ids) + len(valid_ids) + len(test_ids) == n_cells
    return train_ids, valid_ids, test_ids


def save_split_manifest(seed, train_ids, valid_ids, test_ids):
    payload = {
        "protocol_tag": PROTOCOL_TAG,
        "seed": int(seed),
        "warmup_cycles_per_cell": int(WARMUP),
        "train_cells": list(train_ids),
        "valid_cells": list(valid_ids),
        "test_cells": list(test_ids),
        "counts": {
            "train_cells": len(train_ids),
            "valid_cells": len(valid_ids),
            "test_cells": len(test_ids),
        },
    }
    path = os.path.join(SPLIT_DIR, f"MIT_seed{seed}_split.json")
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
    return path


def fit_train_raw_stats(train_ids):
    retained = []
    for cell_id in train_ids:
        x_cell = np.asarray(mit_dataset[cell_id]["features"], dtype=np.float64)
        if len(x_cell) <= WARMUP:
            raise ValueError(f"Cell {cell_id} has only {len(x_cell)} cycles.")
        retained.append(x_cell[WARMUP:])
    stacked = np.vstack(retained)
    return {
        "mean": stacked.mean(axis=0),
        "std": stacked.std(axis=0) + 1e-8,
    }


def prepare_normalized_cells(all_ids, raw_stats):
    normalized = {}
    for cell_id in all_ids:
        x_cell = np.asarray(mit_dataset[cell_id]["features"], dtype=np.float64)
        y_cell = np.asarray(mit_dataset[cell_id]["soh"], dtype=np.float32).reshape(-1)
        if len(x_cell) != len(y_cell):
            raise ValueError(f"Length mismatch for cell {cell_id}: X={len(x_cell)}, y={len(y_cell)}")
        normalized[cell_id] = {
            "X_full": normalize_raw_features(x_cell, raw_stats).astype(np.float32),
            "y_full": y_cell,
        }
    return normalized


def _feature_cache_path(model_label, seed):
    safe = model_label.replace("+", "_").replace(" ", "")
    return os.path.join(FEAT_DIR, f"{safe}_MIT_seed{seed}_{PROTOCOL_TAG}.npz")


def _save_feature_cache(path, payload):
    np.savez_compressed(
        path,
        Rtr=payload["Rtr"],
        ytr=payload["ytr"],
        ctr=payload["ctr"],
        itr=payload["itr"],
        Rv=payload["Rv"],
        yv=payload["yv"],
        cv=payload["cv"],
        iv=payload["iv"],
        Rte=payload["Rte"],
        yte=payload["yte"],
        cte=payload["cte"],
        ite=payload["ite"],
    )


def _load_feature_cache(path):
    z = np.load(path, allow_pickle=True)
    return {key: z[key] for key in z.files}


def generate_split_features(model, normalized_cells, cell_ids, split_name, verbose=True):
    features, labels, cell_labels, cycle_indices = [], [], [], []
    start = time.time()

    for index, cell_id in enumerate(cell_ids):
        x_full = normalized_cells[cell_id]["X_full"]
        y_full = normalized_cells[cell_id]["y_full"]

        # Process all cycles so QRR can build memory, then discard warmup per cell.
        reservoir_full = model.process_cell(x_full)
        reservoir_kept = reservoir_full[WARMUP:]
        y_kept = y_full[WARMUP:]
        idx_kept = np.arange(len(y_full), dtype=np.int32)[WARMUP:]

        if len(reservoir_kept) != len(y_kept):
            raise RuntimeError(f"Post-warmup mismatch for cell {cell_id}")

        features.append(reservoir_kept)
        labels.append(y_kept)
        cell_labels.extend([str(cell_id)] * len(y_kept))
        cycle_indices.append(idx_kept)

        if verbose and ((index + 1) % 10 == 0 or index + 1 == len(cell_ids)):
            elapsed = time.time() - start
            print(
                f"      {model.model_name} {split_name}: "
                f"{index + 1}/{len(cell_ids)} cells, {elapsed:.1f}s"
            )

    return {
        "R": np.vstack(features).astype(np.float32),
        "y": np.concatenate(labels).astype(np.float32),
        "cell": np.asarray(cell_labels, dtype=object),
        "cycle": np.concatenate(cycle_indices).astype(np.int32),
    }


def generate_or_load_all_features(model, seed, normalized_cells, train_ids, valid_ids, test_ids):
    cache_path = _feature_cache_path(model.model_name, seed)
    if USE_FEATURE_CACHE and not REGENERATE_FEATURES and os.path.exists(cache_path):
        log(f"Loading feature cache: {cache_path}")
        return _load_feature_cache(cache_path), 0.0, cache_path

    start = time.time()
    train = generate_split_features(model, normalized_cells, train_ids, "train")
    valid = generate_split_features(model, normalized_cells, valid_ids, "valid")
    test = generate_split_features(model, normalized_cells, test_ids, "test")

    payload = {
        "Rtr": train["R"], "ytr": train["y"], "ctr": train["cell"], "itr": train["cycle"],
        "Rv": valid["R"], "yv": valid["y"], "cv": valid["cell"], "iv": valid["cycle"],
        "Rte": test["R"], "yte": test["y"], "cte": test["cell"], "ite": test["cycle"],
    }
    elapsed = time.time() - start

    if USE_FEATURE_CACHE:
        _save_feature_cache(cache_path, payload)

    return payload, elapsed, cache_path


def assert_paired_feature_metadata(cqrf_payload, qrr_payload):
    for key in ["ytr", "ctr", "itr", "yv", "cv", "iv", "yte", "cte", "ite"]:
        if not np.array_equal(cqrf_payload[key], qrr_payload[key]):
            raise AssertionError(f"Paired metadata mismatch in {key}")
    for key in ["Rtr", "Rv", "Rte"]:
        if cqrf_payload[key].shape != qrr_payload[key].shape:
            raise AssertionError(f"Paired feature shape mismatch in {key}")

In [6]:
# ================================================================
# 6. Shared Tanh CBR training and artifact saving
# ================================================================
class CBRReadout(nn.Module):
    def __init__(self, n_inputs=57, hidden_dims=CBR_HIDDEN_DIMS, dropout=CBR_DROPOUT):
        super().__init__()
        layers = []
        previous = n_inputs
        for hidden in hidden_dims:
            layers.append(nn.Linear(previous, hidden))
            layers.append(nn.Tanh())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous = hidden
        layers.append(nn.Linear(previous, 1))
        self.net = nn.Sequential(*layers)
        self.n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _batched_predict(model, x_array, batch_size=4096):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(x_array), batch_size):
            batch = torch.as_tensor(x_array[start : start + batch_size], dtype=torch.float32, device=device)
            outputs.append(model(batch).detach().cpu().numpy())
    return np.concatenate(outputs)


def train_cbr_paired(
    model_label,
    seed,
    Rtr,
    ytr,
    Rv,
    yv,
    Rte,
    yte,
    test_cells,
    test_cycles,
):
    seed_everything(seed)

    feature_mean = Rtr.mean(axis=0).astype(np.float32)
    feature_std = (Rtr.std(axis=0) + 1e-8).astype(np.float32)

    X_train = ((Rtr - feature_mean) / feature_std).astype(np.float32)
    X_valid = ((Rv - feature_mean) / feature_std).astype(np.float32)
    X_test = ((Rte - feature_mean) / feature_std).astype(np.float32)

    train_tensor = torch.as_tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.as_tensor(ytr, dtype=torch.float32)
    train_dataset = TensorDataset(train_tensor, y_train_tensor)

    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=CBR_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )

    X_valid_t = torch.as_tensor(X_valid, dtype=torch.float32, device=device)
    y_valid_t = torch.as_tensor(yv, dtype=torch.float32, device=device)

    # Reset the seed immediately before model construction so both paired
    # models start from identical CBR weights for the same seed.
    seed_everything(seed)
    model = CBRReadout(n_inputs=Rtr.shape[1]).to(device)
    if model.n_trainable != EXPECTED_CBR_PARAMS:
        raise RuntimeError(f"Expected 125 CBR parameters, got {model.n_trainable}")

    optimizer = optim.Adam(model.parameters(), lr=CBR_LR, weight_decay=CBR_WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=10, factor=0.5, min_lr=1e-6
    )
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = []
    train_start = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(CBR_EPOCHS):
        model.train()
        batch_losses = []

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            predictions = model(x_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CBR_GRAD_CLIP)
            optimizer.step()
            batch_losses.append(float(loss.item()))

        model.eval()
        with torch.no_grad():
            valid_predictions = model(X_valid_t)
            val_loss = float(criterion(valid_predictions, y_valid_t).item())

        train_loss = float(np.mean(batch_losses))
        scheduler.step(val_loss)
        current_lr = float(optimizer.param_groups[0]["lr"])

        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "valid_loss": val_loss,
                "learning_rate": current_lr,
            }
        )

        if val_loss < best_val_loss - 1e-12:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= CBR_PATIENCE:
            break

    train_seconds = time.time() - train_start

    if best_state is None:
        raise RuntimeError("CBR training did not produce a checkpoint.")
    model.load_state_dict(best_state)

    inference_start = time.time()
    test_predictions = _batched_predict(model, X_test)
    inference_seconds = time.time() - inference_start
    metrics = compute_metrics(yte, test_predictions)

    safe_model = model_label.replace("+", "_").replace(" ", "")
    checkpoint_path = os.path.join(CKPT_DIR, f"{safe_model}_MIT_seed{seed}.pt")
    history_path = os.path.join(HIST_DIR, f"{safe_model}_MIT_seed{seed}_history.csv")
    prediction_path = os.path.join(PRED_DIR, f"{safe_model}_MIT_seed{seed}_predictions.csv")

    torch.save(
        {
            "protocol_tag": PROTOCOL_TAG,
            "model_label": model_label,
            "seed": int(seed),
            "state_dict": model.state_dict(),
            "feature_mean": feature_mean,
            "feature_std": feature_std,
            "architecture": [57, 2, 2, 1],
            "activation": "Tanh",
            "dropout": 0.0,
            "n_trainable": int(model.n_trainable),
        },
        checkpoint_path,
    )

    pd.DataFrame(history).to_csv(history_path, index=False)
    pd.DataFrame(
        {
            "cell_id": test_cells.astype(str),
            "cycle_index_zero_based": test_cycles.astype(int),
            "y_true": yte.astype(float),
            "y_pred": test_predictions.astype(float),
            "absolute_error": np.abs(yte - test_predictions).astype(float),
        }
    ).to_csv(prediction_path, index=False)

    peak_cuda_mb = (
        float(torch.cuda.max_memory_allocated(device) / (1024 ** 2))
        if torch.cuda.is_available()
        else 0.0
    )

    training_info = {
        "best_valid_loss": float(best_val_loss),
        "epochs_run": int(len(history)),
        "train_seconds": float(train_seconds),
        "inference_seconds": float(inference_seconds),
        "rss_mb_after_training": current_rss_mb(),
        "peak_cuda_mb": peak_cuda_mb,
        "checkpoint_path": checkpoint_path,
        "history_path": history_path,
        "prediction_path": prediction_path,
        "params": int(model.n_trainable),
    }
    return metrics, training_info

In [7]:
# ================================================================
# 7. Protocol validation before the expensive run
# ================================================================
if len(mit_cells) != 140 or N_INPUTS != 8:
    raise RuntimeError(
        f"Expected 140 MIT cells and 8 inputs, got {len(mit_cells)} cells and {N_INPUTS} inputs."
    )

cqrf_check = CoupledQuantumReservoir10q(n_inputs=N_INPUTS, recurrent=False)
qrr_check = CoupledQuantumReservoir10q(n_inputs=N_INPUTS, recurrent=True)

assert cqrf_check.n_outputs == qrr_check.n_outputs == 57
assert np.array_equal(cqrf_check.reservoir_params, qrr_check.reservoir_params)
assert np.array_equal(cqrf_check.input_projection, qrr_check.input_projection)
assert np.array_equal(cqrf_check.memory_rotation, qrr_check.memory_rotation)

cbr_check = CBRReadout()
assert cbr_check.n_trainable == 125
assert sum(isinstance(layer, nn.Tanh) for layer in cbr_check.net) == 2
assert not any(isinstance(layer, nn.ReLU) for layer in cbr_check.net)

# Synthetic check: first cycle is identical because both variants begin
# from the same zero memory state; later cycles may diverge due recurrence.
synthetic = np.zeros((3, N_INPUTS), dtype=np.float32)
synthetic[:, 0] = [0.1, 0.2, 0.3]
cqrf_synthetic = cqrf_check.process_cell(synthetic)
qrr_synthetic = qrr_check.process_cell(synthetic)
assert np.allclose(cqrf_synthetic[0], qrr_synthetic[0], atol=1e-7)

print("✅ Protocol validation passed")
print("   Reservoir features:", cqrf_check.n_outputs)
print("   Shared CBR parameters:", cbr_check.n_trainable)
print("   Shared activation: Tanh")
print("   QRR resets at each process_cell call")
print("   Active seeds:", ACTIVE_SEEDS)

del cqrf_check, qrr_check, cbr_check
gc.collect()

✅ Protocol validation passed
   Reservoir features: 57
   Shared CBR parameters: 125
   Shared activation: Tanh
   QRR resets at each process_cell call
   Active seeds: [7, 8, 9]


30

In [8]:
# ================================================================
# 8. Paired seed runner
# ================================================================
MASTER_RESULTS_PATH = os.path.join(RES_DIR, "MIT_paired_run_metrics.csv")

def _load_existing_results():
    if os.path.exists(MASTER_RESULTS_PATH):
        return pd.read_csv(MASTER_RESULTS_PATH)
    return pd.DataFrame()


def _is_completed(existing, seed, model_label):
    if existing.empty:
        return False
    mask = (
        (existing["seed"].astype(int) == int(seed))
        & (existing["model"].astype(str) == str(model_label))
        & (existing["protocol_tag"].astype(str) == PROTOCOL_TAG)
    )
    return bool(mask.any())


existing_results = _load_existing_results()
new_rows = []

for seed in ACTIVE_SEEDS:
    log("=" * 90)
    log(f"MIT paired seed {seed}: starting")
    pair_start = time.time()

    train_ids, valid_ids, test_ids = get_mit_split(mit_cells, seed)
    split_path = save_split_manifest(seed, train_ids, valid_ids, test_ids)

    raw_stats = fit_train_raw_stats(train_ids)
    normalized_cells = prepare_normalized_cells(
        train_ids + valid_ids + test_ids,
        raw_stats,
    )

    model_payloads = {}
    model_feature_times = {}
    model_cache_paths = {}

    for recurrent, model_label in [(False, "CQRF+CBR"), (True, "QRR+CBR")]:
        if SKIP_COMPLETED and _is_completed(existing_results, seed, model_label):
            log(f"Seed {seed} {model_label}: already complete; skipping.")
            continue

        reservoir = CoupledQuantumReservoir10q(
            n_inputs=N_INPUTS,
            recurrent=recurrent,
            seed=RESERVOIR_SEED,
            coupling_strength=COUPLING_STRENGTH,
        )

        payload, feature_seconds, cache_path = generate_or_load_all_features(
            reservoir,
            seed,
            normalized_cells,
            train_ids,
            valid_ids,
            test_ids,
        )
        model_payloads[model_label] = payload
        model_feature_times[model_label] = feature_seconds
        model_cache_paths[model_label] = cache_path

        del reservoir
        gc.collect()

    # When both were generated/loaded in this execution, assert exact paired metadata.
    if "CQRF+CBR" in model_payloads and "QRR+CBR" in model_payloads:
        assert_paired_feature_metadata(model_payloads["CQRF+CBR"], model_payloads["QRR+CBR"])

    for model_label, payload in model_payloads.items():
        log(f"Seed {seed} {model_label}: training shared Tanh CBR")

        metrics, train_info = train_cbr_paired(
            model_label=model_label,
            seed=seed,
            Rtr=payload["Rtr"],
            ytr=payload["ytr"],
            Rv=payload["Rv"],
            yv=payload["yv"],
            Rte=payload["Rte"],
            yte=payload["yte"],
            test_cells=payload["cte"],
            test_cycles=payload["ite"],
        )

        row = {
            "protocol_tag": PROTOCOL_TAG,
            "dataset": "MIT",
            "seed": int(seed),
            "model": model_label,
            "activation": "Tanh",
            "readout_architecture": "57-2-2-1",
            "params": int(train_info["params"]),
            "warmup_per_cell": int(WARMUP),
            "train_cells": len(train_ids),
            "valid_cells": len(valid_ids),
            "test_cells": len(test_ids),
            "n_train_cycles": int(len(payload["ytr"])),
            "n_valid_cycles": int(len(payload["yv"])),
            "n_test_cycles": int(len(payload["yte"])),
            "feature_seconds": float(model_feature_times[model_label]),
            "train_seconds": float(train_info["train_seconds"]),
            "inference_seconds": float(train_info["inference_seconds"]),
            "epochs_run": int(train_info["epochs_run"]),
            "best_valid_loss": float(train_info["best_valid_loss"]),
            "rss_mb_after_training": float(train_info["rss_mb_after_training"]),
            "peak_cuda_mb": float(train_info["peak_cuda_mb"]),
            "split_manifest": split_path,
            "feature_cache": model_cache_paths[model_label],
            "checkpoint": train_info["checkpoint_path"],
            "predictions": train_info["prediction_path"],
            "history": train_info["history_path"],
            **metrics,
        }
        new_rows.append(row)
        log(
            f"Seed {seed} {model_label}: "
            f"MAPE={metrics['MAPE']:.6f}, RMSE={metrics['RMSE']:.6f}, "
            f"score={metrics['DerivedScore']:.4f}%"
        )

        # Save after every model so interrupted jobs remain resumable.
        combined = pd.concat(
            [existing_results, pd.DataFrame(new_rows)],
            ignore_index=True,
        )
        combined = combined.drop_duplicates(
            subset=["protocol_tag", "dataset", "seed", "model"],
            keep="last",
        ).sort_values(["seed", "model"])
        combined.to_csv(MASTER_RESULTS_PATH, index=False)

    log(f"MIT paired seed {seed}: finished in {time.time() - pair_start:.1f}s")
    del normalized_cells, model_payloads
    gc.collect()

print("✅ Active seed run complete")
print("Metrics:", MASTER_RESULTS_PATH)



# ================================================================
# 9. Ten-seed summary and paired statistical analysis
# ================================================================
def bootstrap_mean_ci(values, n_resamples=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=np.float64)
    rng = np.random.RandomState(seed)
    sampled_means = np.empty(n_resamples, dtype=np.float64)
    for i in range(n_resamples):
        sample = rng.choice(values, size=len(values), replace=True)
        sampled_means[i] = sample.mean()
    return {
        "lower": float(np.percentile(sampled_means, 2.5)),
        "upper": float(np.percentile(sampled_means, 97.5)),
    }


run_df = pd.read_csv(MASTER_RESULTS_PATH)
run_df = run_df[run_df["protocol_tag"] == PROTOCOL_TAG].copy()

summary = (
    run_df.groupby("model")
    .agg(
        n_seeds=("seed", "nunique"),
        MAPE_mean=("MAPE", "mean"),
        MAPE_std=("MAPE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        score_mean=("DerivedScore", "mean"),
        score_std=("DerivedScore", "std"),
        feature_seconds_mean=("feature_seconds", "mean"),
        train_seconds_mean=("train_seconds", "mean"),
        inference_seconds_mean=("inference_seconds", "mean"),
        rss_mb_mean=("rss_mb_after_training", "mean"),
    )
    .reset_index()
)
summary_path = os.path.join(RES_DIR, "MIT_summary_by_model.csv")
summary.to_csv(summary_path, index=False)

paired = run_df.pivot(index="seed", columns="model", values=["MAPE", "RMSE", "DerivedScore"])
required_models = {"CQRF+CBR", "QRR+CBR"}

paired_stats = {
    "protocol_tag": PROTOCOL_TAG,
    "complete_paired_seeds": 0,
    "status": "insufficient paired seeds",
}

if required_models.issubset(set(run_df["model"].unique())):
    paired_complete = paired.dropna().copy()
    paired_complete.columns = [f"{metric}_{model}" for metric, model in paired_complete.columns]
    paired_complete = paired_complete.reset_index()

    paired_complete["MAPE_difference_CQRF_minus_QRR"] = (
        paired_complete["MAPE_CQRF+CBR"] - paired_complete["MAPE_QRR+CBR"]
    )
    paired_complete["RMSE_difference_CQRF_minus_QRR"] = (
        paired_complete["RMSE_CQRF+CBR"] - paired_complete["RMSE_QRR+CBR"]
    )
    paired_complete["CQRF_win_MAPE"] = (
        paired_complete["MAPE_CQRF+CBR"] < paired_complete["MAPE_QRR+CBR"]
    )

    paired_path = os.path.join(RES_DIR, "MIT_paired_seed_results.csv")
    paired_complete.to_csv(paired_path, index=False)

    differences = paired_complete["MAPE_difference_CQRF_minus_QRR"].to_numpy()
    n_pairs = len(differences)

    if n_pairs >= 2 and not np.allclose(differences, 0):
        wilcoxon_two_sided = wilcoxon(
            paired_complete["MAPE_CQRF+CBR"],
            paired_complete["MAPE_QRR+CBR"],
            alternative="two-sided",
            method="auto",
        )
        wilcoxon_one_sided = wilcoxon(
            paired_complete["MAPE_CQRF+CBR"],
            paired_complete["MAPE_QRR+CBR"],
            alternative="less",
            method="auto",
        )
        two_sided_p = float(wilcoxon_two_sided.pvalue)
        one_sided_p = float(wilcoxon_one_sided.pvalue)
    else:
        two_sided_p = None
        one_sided_p = None

    std_diff = differences.std(ddof=1) if n_pairs > 1 else np.nan
    cohens_dz = float(differences.mean() / std_diff) if np.isfinite(std_diff) and std_diff > 0 else None

    paired_stats = {
        "protocol_tag": PROTOCOL_TAG,
        "status": "complete" if n_pairs == 10 else "partial",
        "complete_paired_seeds": int(n_pairs),
        "seeds": paired_complete["seed"].astype(int).tolist(),
        "mean_MAPE_difference_CQRF_minus_QRR": float(differences.mean()),
        "median_MAPE_difference_CQRF_minus_QRR": float(np.median(differences)),
        "bootstrap_95pct_CI_mean_difference": bootstrap_mean_ci(differences),
        "CQRF_MAPE_wins": int(paired_complete["CQRF_win_MAPE"].sum()),
        "QRR_MAPE_wins": int((~paired_complete["CQRF_win_MAPE"]).sum()),
        "wilcoxon_two_sided_p": two_sided_p,
        "wilcoxon_one_sided_CQRF_less_p": one_sided_p,
        "cohens_dz_CQRF_minus_QRR": cohens_dz,
        "paired_results_csv": paired_path,
    }

stats_path = os.path.join(RES_DIR, "MIT_paired_statistics.json")
with open(stats_path, "w", encoding="utf-8") as handle:
    json.dump(paired_stats, handle, indent=2)

print("\nModel summary")
display(summary)
print("\nPaired statistics")
print(json.dumps(paired_stats, indent=2))
print("\nSaved:")
print(" ", summary_path)
print(" ", stats_path)

[2026-07-13 19:01:11] ==========================================================================================
[2026-07-13 19:01:11] MIT paired seed 7: starting
      CQRF+CBR train: 10/84 cells, 563.5s
      CQRF+CBR train: 20/84 cells, 1027.2s
      CQRF+CBR train: 30/84 cells, 1497.8s
      CQRF+CBR train: 40/84 cells, 1879.1s
      CQRF+CBR train: 50/84 cells, 2212.2s
      CQRF+CBR train: 60/84 cells, 2607.0s
      CQRF+CBR train: 70/84 cells, 2970.9s
      CQRF+CBR train: 80/84 cells, 3363.4s
      CQRF+CBR train: 84/84 cells, 3499.6s
      CQRF+CBR valid: 10/28 cells, 449.9s
      CQRF+CBR valid: 20/28 cells, 848.5s
      CQRF+CBR valid: 28/28 cells, 1147.8s
      CQRF+CBR test: 10/28 cells, 402.8s
      CQRF+CBR test: 20/28 cells, 790.2s
      CQRF+CBR test: 28/28 cells, 1113.2s
      QRR+CBR train: 10/84 cells, 537.8s
      QRR+CBR train: 20/84 cells, 1001.4s
      QRR+CBR train: 30/84 cells, 1472.7s
      QRR+CBR train: 40/84 cells, 1870.4s
      QRR+CBR train: 50/84 cells,

In [9]:
# ================================================================
# 9. Ten-seed summary and paired statistical analysis
# ================================================================
def bootstrap_mean_ci(values, n_resamples=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=np.float64)
    rng = np.random.RandomState(seed)
    sampled_means = np.empty(n_resamples, dtype=np.float64)
    for i in range(n_resamples):
        sample = rng.choice(values, size=len(values), replace=True)
        sampled_means[i] = sample.mean()
    return {
        "lower": float(np.percentile(sampled_means, 2.5)),
        "upper": float(np.percentile(sampled_means, 97.5)),
    }


run_df = pd.read_csv(MASTER_RESULTS_PATH)
run_df = run_df[run_df["protocol_tag"] == PROTOCOL_TAG].copy()

summary = (
    run_df.groupby("model")
    .agg(
        n_seeds=("seed", "nunique"),
        MAPE_mean=("MAPE", "mean"),
        MAPE_std=("MAPE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        score_mean=("DerivedScore", "mean"),
        score_std=("DerivedScore", "std"),
        feature_seconds_mean=("feature_seconds", "mean"),
        train_seconds_mean=("train_seconds", "mean"),
        inference_seconds_mean=("inference_seconds", "mean"),
        rss_mb_mean=("rss_mb_after_training", "mean"),
    )
    .reset_index()
)
summary_path = os.path.join(RES_DIR, "MIT_summary_by_model.csv")
summary.to_csv(summary_path, index=False)

paired = run_df.pivot(index="seed", columns="model", values=["MAPE", "RMSE", "DerivedScore"])
required_models = {"CQRF+CBR", "QRR+CBR"}

paired_stats = {
    "protocol_tag": PROTOCOL_TAG,
    "complete_paired_seeds": 0,
    "status": "insufficient paired seeds",
}

if required_models.issubset(set(run_df["model"].unique())):
    paired_complete = paired.dropna().copy()
    paired_complete.columns = [f"{metric}_{model}" for metric, model in paired_complete.columns]
    paired_complete = paired_complete.reset_index()

    paired_complete["MAPE_difference_CQRF_minus_QRR"] = (
        paired_complete["MAPE_CQRF+CBR"] - paired_complete["MAPE_QRR+CBR"]
    )
    paired_complete["RMSE_difference_CQRF_minus_QRR"] = (
        paired_complete["RMSE_CQRF+CBR"] - paired_complete["RMSE_QRR+CBR"]
    )
    paired_complete["CQRF_win_MAPE"] = (
        paired_complete["MAPE_CQRF+CBR"] < paired_complete["MAPE_QRR+CBR"]
    )

    paired_path = os.path.join(RES_DIR, "MIT_paired_seed_results.csv")
    paired_complete.to_csv(paired_path, index=False)

    differences = paired_complete["MAPE_difference_CQRF_minus_QRR"].to_numpy()
    n_pairs = len(differences)

    if n_pairs >= 2 and not np.allclose(differences, 0):
        wilcoxon_two_sided = wilcoxon(
            paired_complete["MAPE_CQRF+CBR"],
            paired_complete["MAPE_QRR+CBR"],
            alternative="two-sided",
            method="auto",
        )
        wilcoxon_one_sided = wilcoxon(
            paired_complete["MAPE_CQRF+CBR"],
            paired_complete["MAPE_QRR+CBR"],
            alternative="less",
            method="auto",
        )
        two_sided_p = float(wilcoxon_two_sided.pvalue)
        one_sided_p = float(wilcoxon_one_sided.pvalue)
    else:
        two_sided_p = None
        one_sided_p = None

    std_diff = differences.std(ddof=1) if n_pairs > 1 else np.nan
    cohens_dz = float(differences.mean() / std_diff) if np.isfinite(std_diff) and std_diff > 0 else None

    paired_stats = {
        "protocol_tag": PROTOCOL_TAG,
        "status": "complete" if n_pairs == 10 else "partial",
        "complete_paired_seeds": int(n_pairs),
        "seeds": paired_complete["seed"].astype(int).tolist(),
        "mean_MAPE_difference_CQRF_minus_QRR": float(differences.mean()),
        "median_MAPE_difference_CQRF_minus_QRR": float(np.median(differences)),
        "bootstrap_95pct_CI_mean_difference": bootstrap_mean_ci(differences),
        "CQRF_MAPE_wins": int(paired_complete["CQRF_win_MAPE"].sum()),
        "QRR_MAPE_wins": int((~paired_complete["CQRF_win_MAPE"]).sum()),
        "wilcoxon_two_sided_p": two_sided_p,
        "wilcoxon_one_sided_CQRF_less_p": one_sided_p,
        "cohens_dz_CQRF_minus_QRR": cohens_dz,
        "paired_results_csv": paired_path,
    }

stats_path = os.path.join(RES_DIR, "MIT_paired_statistics.json")
with open(stats_path, "w", encoding="utf-8") as handle:
    json.dump(paired_stats, handle, indent=2)

print("\nModel summary")
display(summary)
print("\nPaired statistics")
print(json.dumps(paired_stats, indent=2))
print("\nSaved:")
print(" ", summary_path)
print(" ", stats_path)


Model summary


,model,n_seeds,MAPE_mean,MAPE_std,RMSE_mean,RMSE_std,score_mean,score_std,feature_seconds_mean,train_seconds_mean,inference_seconds_mean,rss_mb_mean
0,CQRF+CBR,3,0.013200,0.000370,0.019571,0.001840,98.679999,0.037037,5766.313309,121.282493,0.002975,821.031250
1,QRR+CBR,3,0.015203,0.001145,0.022868,0.003334,98.479731,0.114497,5767.794951,135.074364,0.003394,821.869792



Paired statistics
{
  "protocol_tag": "paired_v1_per_cell_warmup_external_val_tanh",
  "status": "partial",
  "complete_paired_seeds": 3,
  "seeds": [
    7,
    8,
    9
  ],
  "mean_MAPE_difference_CQRF_minus_QRR": -0.002002679438714633,
  "median_MAPE_difference_CQRF_minus_QRR": -0.0022386024401258014,
  "bootstrap_95pct_CI_mean_difference": {
    "lower": -0.0029374836375025987,
    "upper": -0.0008319522385154994
  },
  "CQRF_MAPE_wins": 3,
  "QRR_MAPE_wins": 0,
  "wilcoxon_two_sided_p": 0.25,
  "wilcoxon_one_sided_CQRF_less_p": 0.125,
  "cohens_dz_CQRF_minus_QRR": -1.8674591939433787,
  "paired_results_csv": "/content/drive/MyDrive/HB/QRC_Battery_SOH/results/789_EXP-MIT-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-v1/MIT_paired_seed_results.csv"
}

Saved:
  /content/drive/MyDrive/HB/QRC_Battery_SOH/results/789_EXP-MIT-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-v1/MIT_summary_by_model.csv
  /content/drive/MyDrive/HB/QRC_Battery_SOH/results/789_EXP-MIT-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-v1/MIT_paired_sta

## Execution sequence

1. Run all cells through **Protocol validation**.
2. Keep `RUN_MODE = "SMOKE"` and execute seed 0.
3. Inspect the seed-0 outputs:
   - both models must have 125 parameters;
   - train/validation/test cycle counts must match;
   - both prediction files must contain identical test cell IDs and cycle indices;
   - QRR must reset at every cell boundary.
4. Change `RUN_MODE = "FULL"`.
5. Re-run the configuration and runner cells. Completed model/seed pairs are skipped when `SKIP_COMPLETED=True`.
6. Run the summary/statistics cell after all ten paired seeds finish.

A negative value of `MAPE_difference_CQRF_minus_QRR` favors CQRF.